In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

In [4]:
# we first note the different labels present for the X-rays in the MIMIC-CXR-JPG dataset
LABELS = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
          'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
          'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other',
          'Pneumonia', 'Pneumothorax', 'Support Devices']

IMG_DIR = r"D:\omer files\projects\NMIMS\cxr_project"

In [5]:
class CXRDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        def build_path(row):
            subj = f"p{str(row['subject_id'])[:2]}/p{row['subject_id']}"
            study = f"s{row['study_id']}"
            return os.path.join('files', subj, study, f"{row['dicom_id']}.jpg")
        self.df['rel_path'] = self.df.apply(build_path, axis=1)

        self.df['full_path'] = self.df['rel_path'].apply(lambda p: os.path.join(img_dir, p))
        before = len(self.df)
        self.df = self.df[self.df['full_path'].apply(os.path.exists)].reset_index(drop=True)
        print(f"{csv_path}: {len(self.df)}/{before} images found on disk")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(row[LABELS].values.astype('float32'))
        return img, labels

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness = 0.1, contrast = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [7]:
train_ds = CXRDataset('subset_train.csv', IMG_DIR, transform=train_transform)
val_ds = CXRDataset('subset_val.csv', IMG_DIR, transform=eval_transform)
test_ds = CXRDataset('subset_test.csv', IMG_DIR, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_ds, batch_size = 32, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_ds, batch_size = 32, shuffle = False, num_workers = 0)

subset_train.csv: 5537/5537 images found on disk
subset_val.csv: 300/300 images found on disk
subset_test.csv: 300/300 images found on disk


In [8]:
imgs, labels = next(iter(train_loader))
print(f'Batch image shape: {imgs.shape}')
print(f'Batch label shape: {labels.shape}')
print(f'Label sum per sample (first 5): {labels[:5].sum(dim = 1)}')

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32, 14])
Label sum per sample (first 5): tensor([6., 2., 4., 4., 3.])
